# ParkVision · EfficientNetB0: vehicle present / no vehicle

**Upload this `.ipynb` into Google Colab, select a T4 GPU, then Run all.**

This notebook trains **only EfficientNetB0**. It exports a `.keras` file that fits
the existing ParkVision worker without changing the database or input shape.

**The exact rule:** one supplied parking-box crop → target vehicle visible =
`occupied` (1); no target vehicle = `unoccupied` (0, mapped to `vacant` in the app).
Cars/SUVs/vans/pickups are covered by COCO's car/truck categories; bus is included.
Add `motorcycle` to `VEHICLE_CATEGORIES` if it should count too.

No vehicle does **not** prove a space is physically clear or large enough to park.
People, boxes and cones are negatives for this binary task; obstruction detection
is outside this notebook. This revises the target from general occupancy to
vehicle presence. Do not claim the original three-class requirement is complete.

**What changed from S2:** broader vehicle/no-vehicle examples, annotation-checked background
negatives, grouped splits, source/class balancing, actual backbone fine-tuning,
Drive checkpoints, an untouched test set, and an augmentation-free export.

Public datasets can start the run immediately. To address the real Luzano night
errors, add representative labeled crops from the mounted phone. No notebook can
promise correct predictions on unseen cameras without measuring them.

## 1 · Runtime and installation

Use **Runtime → Change runtime type → T4 GPU**. Run from the top in a fresh
session. The first cells install matching TensorFlow/Keras versions. Drive access
is requested once so checkpoints survive disconnection. The notebook downloads
public data in Colab, including a 253 MB COCO annotation archive plus selected
images and the PKLot download; allow several GB of disk space.

In [ ]:
import sys, subprocess
PACKAGES = [
    'tensorflow==2.20.0', 'keras==3.13.2', 'Pillow>=11.3,<13',
    'pandas>=2.2,<3', 'scikit-learn>=1.5,<2', 'matplotlib>=3.9,<4',
    'kagglehub>=0.3,<1', 'requests>=2.32,<3', 'tqdm>=4.66,<5',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *PACKAGES])
print('Dependencies installed. Continue downward in this session.')

## 2 · Settings

Defaults train on PKLot crops plus COCO vehicle/background crops. No model API
key is needed. `UPLOAD_SITE_ZIP=True` adds your camera-specific crops using the
folder format shown in section 5. Leave it False for a public-data baseline.
Sampling caps control runtime, not the real dataset size. Class 0/1 order is fixed.

In [ ]:
CONFIG = {
    'SEED': 42,
    'REQUIRE_GPU': True,
    'USE_GOOGLE_DRIVE': True,
    'USE_PKLOT': True,
    'USE_COCO': True,
    'PKLOT_ROOT': '',  # Existing Colab dataset path, or empty to use kagglehub.
    'UPLOAD_SITE_ZIP': False,
    'SITE_ROOT': '',   # Existing extracted local-crop directory instead of ZIP upload.
    'VEHICLE_CATEGORIES': ['car', 'truck', 'bus'],
    'PKLOT_PER_CLASS': {'train': 3500, 'val': 600, 'test': 600},
    'COCO_PER_CLASS': {'train': 700, 'val': 150, 'test': 150},
    'SOURCE_WEIGHTS': {'pklot': 1.0, 'coco': 2.0, 'site': 4.0},
    'MAX_TRAIN_SAMPLES_PER_EPOCH': 16000,
    'BATCH_SIZE': 32,
    'HEAD_EPOCHS': 5,
    'FINE_TUNE_EPOCHS': 10,
    'RUN_MANUAL_IMAGE_TEST': False,
}

In [ ]:
import os, json, random, hashlib, zipfile, math
from pathlib import Path
from datetime import datetime, timezone
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from PIL import Image, ImageOps
from IPython.display import display

tf.keras.utils.set_random_seed(CONFIG['SEED'])
gpus = tf.config.list_physical_devices('GPU')
print('TensorFlow:', tf.__version__, 'Keras:', tf.keras.__version__, 'GPU:', gpus)
if CONFIG['REQUIRE_GPU'] and not gpus:
    raise RuntimeError('Select Runtime → Change runtime type → T4 GPU, then Run all again. No training has started.')
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass
if CONFIG['USE_GOOGLE_DRIVE']:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = Path('/content/drive/MyDrive/ParkVision_EfficientNetB0')
else:
    PROJECT_DIR = Path('/content/ParkVision_EfficientNetB0')
    print('WARNING: Drive is disabled. Download exports before this temporary runtime ends.')
CACHE_DIR = Path('/content/parkvision_data_cache')
CROP_DIR = CACHE_DIR / 'crops'
RUN_DIR = PROJECT_DIR / 'runs' / datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
for directory in (CACHE_DIR, CROP_DIR, RUN_DIR):
    directory.mkdir(parents=True, exist_ok=True)
(RUN_DIR / 'config.json').write_text(json.dumps(CONFIG, indent=2))
print('Checkpoints and reports:', RUN_DIR)
print('0 = unoccupied / app vacant; 1 = occupied. Inputs stay RGB 0–255.')

## 3 · Data preparation helpers (run this cell unchanged)

The following code is embedded in this notebook. You do not need to upload the
ParkVision repository. PKLot labels come from category names, not guessed IDs.
Whole capture-day/source-image groups are assigned before cropping. COCO negative
crops cannot overlap an annotated target vehicle. Inspect the sample images:
public annotations can still have mistakes, especially for tiny/partly hidden cars.
Empty PKLot crops that intersect an occupied-position box are excluded because
their labels would be ambiguous for the vehicle-presence target.

In [ ]:
"""Training-data utilities embedded into the standalone ParkVision Colab notebook."""
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from functools import lru_cache
from pathlib import Path
import hashlib
import json
import math
import random
import re
import shutil
import zipfile

import numpy as np
import pandas as pd
from PIL import Image, ImageOps
import requests
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from tqdm.auto import tqdm

SPLITS = ('train', 'val', 'test')
CLASS_NAMES = ['unoccupied', 'occupied']  # Explicit numeric order; never inferred from folders.
IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png', '.webp'}


def clean_bbox(box, image_width, image_height, min_size=8):
    if not isinstance(box, (list, tuple)) or len(box) != 4:
        return None
    if any(isinstance(x, bool) or not isinstance(x, (int, float)) or not math.isfinite(x) for x in box):
        return None
    x, y, width, height = box
    if width <= 0 or height <= 0:
        return None
    left, top = max(0, math.floor(x)), max(0, math.floor(y))
    right, bottom = min(image_width, math.ceil(x + width)), min(image_height, math.ceil(y + height))
    return (left, top, right, bottom) if right - left >= min_size and bottom - top >= min_size else None


def overlaps(a, b):
    return min(a[2], b[2]) > max(a[0], b[0]) and min(a[3], b[3]) > max(a[1], b[1])


def negative_crop(width, height, vehicle_boxes, rng, attempts=60):
    """Never label a proposed negative if it overlaps an annotated target vehicle."""
    for _ in range(attempts):
        cw = max(16, round(width * rng.uniform(0.22, 0.75)))
        ch = max(16, round(height * rng.uniform(0.22, 0.75)))
        if cw > width or ch > height:
            continue
        x, y = rng.randint(0, width - cw), rng.randint(0, height - ch)
        box = (x, y, x + cw, y + ch)
        if not any(overlaps(box, vehicle) for vehicle in vehicle_boxes):
            return box
    return None


def frame_group(filename):
    """Keep PKLot capture days together when dates are recoverable from names."""
    name = Path(filename).name
    date = re.search(r'(20\d{2})[-_](\d{2})[-_](\d{2})', name)
    if date:
        return 'pklot_day_' + '-'.join(date.groups()), 'capture_day'
    stem = re.sub(r'\.rf\.[^.]+', '', Path(name).stem)
    return 'pklot_frame_' + stem, 'source_frame_only'


def grouped_split(frame, group_column='group_id', seed=42):
    """Assign complete groups before generating crops, with train/val/test isolation."""
    frame = frame.reset_index(drop=True).copy()
    if frame[group_column].nunique() < 3:
        raise ValueError('Need at least three independent capture-day/recording groups for train, val and test.')
    splitter = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=seed)
    train_idx, holdout_idx = next(splitter.split(frame, groups=frame[group_column]))
    holdout = frame.iloc[holdout_idx]
    if holdout[group_column].nunique() < 2:
        groups = sorted(frame[group_column].unique())
        random.Random(seed).shuffle(groups)
        mapping = {group: 'train' for group in groups}
        mapping[groups[-1]], mapping[groups[-2]] = 'test', 'val'
        frame['split'] = frame[group_column].map(mapping)
    else:
        second = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=seed + 1)
        val_idx, test_idx = next(second.split(holdout, groups=holdout[group_column]))
        frame['split'] = 'train'
        frame.loc[holdout.iloc[val_idx].index, 'split'] = 'val'
        frame.loc[holdout.iloc[test_idx].index, 'split'] = 'test'
    assert frame.groupby(group_column)['split'].nunique().max() == 1
    return frame


@lru_cache(maxsize=20000)
def file_hash(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def save_crop(image, box, directory, *, source, group_id, split, label, source_id, source_sha):
    crop = image.crop(box).convert('RGB')
    pixel_sha = hashlib.sha256(str(crop.size).encode() + crop.tobytes()).hexdigest()
    directory = Path(directory) / source / split / CLASS_NAMES[label]
    directory.mkdir(parents=True, exist_ok=True)
    target = directory / f'{pixel_sha}.jpg'
    if not target.exists():
        crop.save(target, quality=95, subsampling=0)
    return dict(path=str(target), label=int(label), source=source, group_id=group_id,
                split=split, source_id=str(source_id), source_sha=source_sha, crop_sha=pixel_sha)


def download_public(url, target, max_bytes=400_000_000):
    target = Path(target)
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists() and target.stat().st_size > 0:
        return target
    for attempt in range(3):
        partial = target.with_suffix(target.suffix + '.part')
        try:
            with requests.get(url, stream=True, timeout=(20, 90)) as response:
                response.raise_for_status()
                size = 0
                with partial.open('wb') as output:
                    for chunk in response.iter_content(1024 * 1024):
                        if chunk:
                            size += len(chunk)
                            if size > max_bytes:
                                raise ValueError(f'Download exceeds the configured size limit: {url}')
                            output.write(chunk)
                if not size:
                    raise ValueError('Empty download')
            partial.replace(target)
            return target
        except (requests.RequestException, OSError):
            partial.unlink(missing_ok=True)
            if attempt == 2:
                raise


def safe_extract_zip(archive, target, max_bytes=5_000_000_000):
    target = Path(target).resolve()
    target.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as bundle:
        if sum(item.file_size for item in bundle.infolist()) > max_bytes:
            raise ValueError('Archive expands beyond the configured size limit.')
        for item in bundle.infolist():
            destination = (target / item.filename).resolve()
            if not destination.is_relative_to(target):
                raise ValueError('Archive contains an unsafe path.')
        bundle.extractall(target)
    return target


def build_pklot(root, crop_dir, per_class, seed=42):
    root = Path(root)
    by_basename = defaultdict(list)
    for path in root.rglob('*'):
        if path.suffix.lower() in IMAGE_SUFFIXES:
            by_basename[path.name].append(path)
    records = []
    ignored = Counter()
    for annotation_file in sorted(root.rglob('*.json')):
        if 'annotation' not in annotation_file.name.lower():
            continue
        data = json.loads(annotation_file.read_text())
        if not all(key in data for key in ('images', 'annotations', 'categories')):
            continue
        names = {item['id']: str(item['name']).lower().strip().replace('_', '-').replace(' ', '-') for item in data['categories']}
        categories = {}
        for category_id, name in names.items():
            if name in {'space-empty', 'empty', 'vacant', 'unoccupied', 'free'}:
                categories[category_id] = 0
            elif name in {'space-occupied', 'occupied', 'busy'}:
                categories[category_id] = 1
        if set(categories.values()) != {0, 1}:
            raise ValueError(f'Unrecognized PKLot category names: {names}. Check the labels explicitly.')
        annotations = defaultdict(list)
        for annotation in data['annotations']:
            if annotation.get('category_id') in categories:
                annotations[annotation['image_id']].append(annotation)
        rng = random.Random(seed)
        for image in data['images']:
            relative = image['file_name']
            path = (annotation_file.parent / relative).resolve()
            if not path.is_relative_to(root.resolve()) or not path.is_file():
                alternatives = by_basename[Path(relative).name]
                if len(alternatives) != 1:
                    ignored['missing_or_ambiguous_image'] += 1
                    continue
                path = alternatives[0]
            candidates = annotations[image['id']][:]
            occupied_boxes = [clean_bbox(item.get('bbox'), image['width'], image['height'], min_size=1)
                              for item in candidates if categories[item['category_id']] == 1]
            occupied_boxes = [box for box in occupied_boxes if box is not None]
            rng.shuffle(candidates)
            counts = Counter()
            group, grouping_rule = frame_group(relative)
            for annotation in candidates:
                label = categories[annotation['category_id']]
                if counts[label] >= 2:  # More source frames, fewer correlated crops per frame.
                    continue
                box = clean_bbox(annotation.get('bbox'), image['width'], image['height'])
                if box is None:
                    ignored['invalid_box'] += 1
                    continue
                # PKLot labels parking positions, rather than every visible car.
                # Drop an empty position whose rectangular crop also intersects
                # an occupied position: that is ambiguous for vehicle presence.
                if label == 0 and any(overlaps(box, occupied) for occupied in occupied_boxes):
                    ignored['empty_crop_overlaps_occupied_position'] += 1
                    continue
                counts[label] += 1
                records.append(dict(source_path=str(path), box=box, label=label, group_id=group,
                                    grouping_rule=grouping_rule, source_id=path.name))
    if not records:
        raise ValueError('No usable PKLot COCO annotations found. Set PKLOT_ROOT to the dataset directory.')
    frame = grouped_split(pd.DataFrame(records), seed=seed)
    output, kept_counts = [], Counter()
    for split in SPLITS:
        for label in (0, 1):
            candidates = frame[(frame.split == split) & (frame.label == label)].sample(frac=1, random_state=seed)
            for row in tqdm(candidates.to_dict('records'), desc=f'PKLot {split} {CLASS_NAMES[label]}', leave=False):
                if kept_counts[split, label] >= per_class[split]:
                    break
                try:
                    with Image.open(row['source_path']) as image:
                        image = image.convert('RGB')
                        if row['box'][2] > image.width or row['box'][3] > image.height:
                            ignored['image_annotation_size_mismatch'] += 1
                            continue
                        item = save_crop(image, row['box'], crop_dir, source='pklot', group_id=row['group_id'],
                                         split=split, label=label, source_id=row['source_id'], source_sha=file_hash(row['source_path']))
                        item['grouping_rule'] = row['grouping_rule']
                        output.append(item)
                        kept_counts[split, label] += 1
                except (OSError, ValueError):
                    ignored['unreadable_image'] += 1
    print('PKLot ignored records:', dict(ignored))
    return pd.DataFrame(output)


def build_coco(cache_dir, crop_dir, per_class, vehicle_categories=('car', 'truck', 'bus'), seed=42):
    cache_dir = Path(cache_dir) / 'coco'
    # Official COCO S3 bucket alias uses a valid HTTPS certificate.
    base_url = 'https://s3.amazonaws.com/images.cocodataset.org'
    archive = download_public(base_url + '/annotations/annotations_trainval2017.zip', cache_dir / 'annotations.zip')
    annotation_path = cache_dir / 'instances_val2017.json'
    if not annotation_path.exists():
        with zipfile.ZipFile(archive) as bundle:
            with bundle.open('annotations/instances_val2017.json') as source, annotation_path.open('wb') as target:
                shutil.copyfileobj(source, target)
    data = json.loads(annotation_path.read_text())
    target_ids = {item['id'] for item in data['categories'] if item['name'] in vehicle_categories}
    if len(target_ids) != len(vehicle_categories):
        raise ValueError('Unknown COCO vehicle category. Check VEHICLE_CATEGORIES.')
    annotations = defaultdict(list)
    for annotation in data['annotations']:
        if annotation['category_id'] in target_ids:
            annotations[annotation['image_id']].append(annotation)
    images = pd.DataFrame(data['images'])
    images['has_vehicle'] = images.id.map(lambda image_id: bool(annotations[image_id])).astype(int)
    train, holdout = train_test_split(images, test_size=0.30, random_state=seed, stratify=images.has_vehicle)
    val, test = train_test_split(holdout, test_size=0.5, random_state=seed + 1, stratify=holdout.has_vehicle)
    selected = []
    for split, frame in [('train', train), ('val', val), ('test', test)]:
        for label in (0, 1):
            pool = frame[frame.has_vehicle == label].sample(frac=1, random_state=seed)
            # Reserve some extra images in case a positive annotation is too small/corrupt.
            pool = pool.head(min(len(pool), math.ceil(per_class[split] * 1.3)))
            selected.extend({**item, 'split': split} for item in pool.to_dict('records'))
    def fetch(record):
        name = Path(record['file_name']).name
        target = cache_dir / 'images' / name
        try:
            download_public(f'{base_url}/val2017/{name}', target, max_bytes=30_000_000)
            return record, target
        except (requests.RequestException, OSError, ValueError):
            return record, None
    paths = {}
    with ThreadPoolExecutor(max_workers=8) as executor:
        jobs = [executor.submit(fetch, record) for record in selected]
        for job in tqdm(as_completed(jobs), total=len(jobs), desc='COCO image downloads'):
            record, path = job.result()
            paths[record['id']] = path
    output, counts = [], Counter()
    rng = random.Random(seed)
    # Vehicle scenes first so their verified background patches become hard negatives.
    selected.sort(key=lambda item: (-item['has_vehicle'], item['id']))
    for record in tqdm(selected, desc='COCO vehicle / no-vehicle crops'):
        path, split = paths[record['id']], record['split']
        if path is None:
            continue
        try:
            with Image.open(path) as original:
                image = original.convert('RGB')
                boxes = [clean_bbox(item['bbox'], image.width, image.height, min_size=1) for item in annotations[record['id']]]
                boxes = [box for box in boxes if box is not None]
                candidates = []
                if record['has_vehicle'] and boxes:
                    # Prefer a clearly visible vehicle, retain varied surrounding context.
                    box = max(boxes, key=lambda b: (b[2] - b[0]) * (b[3] - b[1]))
                    if (box[2] - box[0]) * (box[3] - box[1]) >= 400:
                        padding = rng.uniform(0.12, 0.65)
                        bw, bh = box[2] - box[0], box[3] - box[1]
                        expanded = (max(0, math.floor(box[0] - bw * padding)), max(0, math.floor(box[1] - bh * padding)),
                                    min(image.width, math.ceil(box[2] + bw * padding)), min(image.height, math.ceil(box[3] + bh * padding)))
                        candidates.append((1, expanded))
                    background = negative_crop(image.width, image.height, boxes, rng)
                    # Keep room for vehicle-free scenes too; do not fill every
                    # negative slot from vehicle-scene backgrounds alone.
                    if background and counts[split, 0] < math.ceil(per_class[split] / 2):
                        candidates.append((0, background))
                elif not record['has_vehicle']:
                    box = (0, 0, image.width, image.height) if rng.random() < 0.5 else negative_crop(image.width, image.height, [], rng)
                    if box:
                        candidates.append((0, box))
                for label, box in candidates:
                    if counts[split, label] >= per_class[split]:
                        continue
                    output.append(save_crop(image, box, crop_dir, source='coco', group_id=f"coco_val2017_{record['id']}",
                                            split=split, label=label, source_id=record['file_name'], source_sha=file_hash(str(path))))
                    counts[split, label] += 1
        except (OSError, ValueError):
            continue
    if not output:
        raise ValueError('COCO download/crop generation produced no data. Check the download error and retry.')
    return pd.DataFrame(output)


def build_local_crops(root, crop_dir, seed=42):
    """Input: site/day_or_recording_id/{occupied,unoccupied}/*.jpg."""
    root = Path(root)
    rows = []
    labels = {'unoccupied': 0, 'vacant': 0, 'empty': 0, 'occupied': 1}
    for path in root.rglob('*'):
        if path.suffix.lower() not in IMAGE_SUFFIXES or path.parent.name.lower() not in labels:
            continue
        group_folder = path.parent.parent
        if group_folder.resolve() == root.resolve():
            raise ValueError('Local data needs recording/day folders above the class folders. Do not randomly split neighboring video frames.')
        rows.append(dict(source_path=str(path), label=labels[path.parent.name.lower()], group_id='site_' + str(group_folder.relative_to(root))))
    if not rows:
        raise ValueError('No local crops found. Use day01/occupied and day01/unoccupied folders inside the ZIP.')
    frame = grouped_split(pd.DataFrame(rows), seed=seed)
    output = []
    for row in frame.to_dict('records'):
        with Image.open(row['source_path']) as source:
            image = ImageOps.exif_transpose(source).convert('RGB')
            if min(image.size) < 16:
                raise ValueError(f"Local crop is too small: {row['source_path']}")
            output.append(save_crop(image, (0, 0, image.width, image.height), crop_dir, source='site', group_id=row['group_id'],
                                    split=row['split'], label=row['label'], source_id=Path(row['source_path']).name, source_sha=file_hash(row['source_path'])))
    return pd.DataFrame(output)


def audit_manifest(frame):
    frame = frame.copy().reset_index(drop=True)
    if not set(frame.label.unique()) <= {0, 1} or set(frame.split.unique()) != set(SPLITS):
        raise ValueError('Need explicit binary labels and train/val/test splits.')
    collisions = frame.groupby('group_id')['split'].nunique()
    if (collisions > 1).any():
        raise ValueError('A recording/source group appears in more than one split.')
    conflicts = set(frame.groupby('crop_sha')['label'].nunique().loc[lambda values: values > 1].index)
    removed = frame[frame.crop_sha.isin(conflicts)].assign(reason='conflicting_labels')
    frame = frame[~frame.crop_sha.isin(conflicts)].copy()
    priority = {'test': 0, 'val': 1, 'train': 2}
    frame['_priority'] = frame.split.map(priority)
    for column in ('source_sha', 'crop_sha'):
        keep_split = frame.sort_values('_priority').drop_duplicates(column).set_index(column)['split']
        mask = frame.split != frame[column].map(keep_split)
        removed = pd.concat([removed, frame[mask].assign(reason=f'{column}_cross_split')], ignore_index=True)
        frame = frame[~mask].copy()
    frame = frame.drop_duplicates('crop_sha').drop(columns='_priority').reset_index(drop=True)
    for split in SPLITS:
        counts = frame[frame.split == split].label.value_counts()
        if any(counts.get(label, 0) < 5 for label in (0, 1)):
            raise ValueError(f'{split} needs at least five distinct crops per class after leakage cleaning; collect more groups.')
    for source in frame.source.unique():
        for split in SPLITS:
            counts = frame[(frame.source == source) & (frame.split == split)].label.value_counts()
            if any(counts.get(label, 0) < 5 for label in (0, 1)):
                raise ValueError(f'{source}/{split} needs at least five unique crops of each class. Add recording groups containing both classes.')
    assert frame.groupby('source_sha')['split'].nunique().max() == 1
    assert frame.groupby('crop_sha')['split'].nunique().max() == 1
    return frame, removed.drop(columns='_priority', errors='ignore')

## 4 · Build the public-data baseline

PKLot supplies actual empty/occupied parking positions. COCO adds different car
angles and vehicle-free backgrounds, including negative patches from car scenes.
The COCO **val2017 source pool is repartitioned** here into this experiment's own
train/val/test groups; it is not being presented as the official COCO benchmark.
PKLot is also regrouped by capture day when filenames permit, instead of retaining
a potentially frame-random third-party split. All resulting assignments are saved.

In [ ]:
sources = []
if CONFIG['USE_PKLOT']:
    if CONFIG['PKLOT_ROOT']:
        pklot_root = Path(CONFIG['PKLOT_ROOT'])
    else:
        import kagglehub
        pklot_root = Path(kagglehub.dataset_download('ammarnassanalhajali/pklot-dataset'))
    print('PKLot directory:', pklot_root)
    pklot = build_pklot(pklot_root, CROP_DIR, CONFIG['PKLOT_PER_CLASS'], CONFIG['SEED'])
    sources.append(pklot)
    display(pklot.groupby(['split', 'label']).size().unstack(fill_value=0))
    print('PKLot grouping methods:', pklot.grouping_rule.value_counts().to_dict())
if CONFIG['USE_COCO']:
    coco = build_coco(CACHE_DIR, CROP_DIR, CONFIG['COCO_PER_CLASS'], tuple(CONFIG['VEHICLE_CATEGORIES']), CONFIG['SEED'])
    sources.append(coco)
    display(coco.groupby(['split', 'label']).size().unstack(fill_value=0))

## 5 · Optional: add your mounted-phone crops

Set `UPLOAD_SITE_ZIP=True` above to choose a ZIP, or set `SITE_ROOT` to an existing
extracted folder. Each image must already be a **crop of one intended parking
position**, using the same box when occupied/empty. Include empty floor, walls,
shadows, building ceiling where unavoidable, night/day lighting, cars of different
colors, and partly visible cars. Do not redraw a special favorable box for each car.

Organize independent capture **days/sessions** like this (at least three groups):

```text
site_crops/
  day01/
    occupied/       # Vehicle visible in that parking crop.
      frame_001.jpg
    unoccupied/     # No target vehicle in the crop.
      frame_002.jpg
  day02/
    occupied/...
    unoccupied/...
  day03/
    occupied/...
    unoccupied/...
```

Keep neighboring frames and all clips from one capture session in the same group.
More independent groups are better than thousands of nearly identical frames.
Two photos are a useful sanity check, not enough to measure deployment accuracy.
Local training and held-out groups must contain both classes. Site data receives
more sampling weight when present; no local accuracy is claimed when it is absent.

In [ ]:
site_root = CONFIG['SITE_ROOT']
if CONFIG['UPLOAD_SITE_ZIP']:
    from google.colab import files
    uploaded = files.upload()
    archives = [name for name in uploaded if name.lower().endswith('.zip')]
    if len(archives) != 1:
        raise ValueError('Upload exactly one ZIP of labeled site crops.')
    site_root = str(safe_extract_zip(archives[0], CACHE_DIR / 'site_upload'))
if site_root:
    site = build_local_crops(site_root, CROP_DIR, CONFIG['SEED'])
    sources.append(site)
    display(site.groupby(['split', 'label']).size().unstack(fill_value=0))
else:
    print('No site crops supplied. This run is a public-data candidate, not a validated Luzano model.')
if not sources:
    raise ValueError('Enable a public dataset or supply local crops before training.')

## 6 · Leakage/label audit and inspect actual inputs

Conflicting exact crop labels are excluded. Duplicate source images/crops cannot
cross splits: holdout data is retained in preference to training copies. This does
not prove all near-duplicates are gone; grouping by real capture sessions remains
important. If a split lacks enough unique examples, the notebook stops rather than
reporting a misleading evaluation.

In [ ]:
manifest, excluded = audit_manifest(pd.concat(sources, ignore_index=True))
manifest.to_csv(RUN_DIR / 'manifest.csv', index=False)
excluded.to_csv(RUN_DIR / 'excluded_records.csv', index=False)
print('Retained:', len(manifest), 'Excluded conflicting/leaking rows:', len(excluded))
display(manifest.groupby(['source', 'split', 'label']).size().rename('examples').reset_index())
assert manifest.groupby('group_id')['split'].nunique().max() == 1
assert manifest.groupby('source_sha')['split'].nunique().max() == 1
assert manifest.groupby('crop_sha')['split'].nunique().max() == 1

train_examples = manifest[manifest.split == 'train']
examples_per_class = min(8, int(train_examples.label.value_counts().min()))
sample = train_examples.groupby('label', group_keys=False).sample(n=examples_per_class, random_state=CONFIG['SEED'])
fig, axes = plt.subplots(4, 4, figsize=(11, 11))
for axis in axes.ravel():
    axis.axis('off')
for axis, row in zip(axes.ravel(), sample.to_dict('records')):
    with Image.open(row['path']) as image:
        axis.imshow(image)
    axis.set_title(f"{row['source']} · {CLASS_NAMES[row['label']]}", fontsize=9)
    axis.axis('off')
fig.suptitle('Inspect labels BEFORE training; do not relabel errors just to improve scores')
fig.tight_layout(); plt.show()

## 7 · EfficientNetB0 model and datasets

Pixels remain RGB float32 in **0–255**; EfficientNet includes its own scaling.
Only training sees augmentation. Classes and sources are balanced through training
sampling, while validation loss is source-balanced so a small site dataset is not
drowned out by PKLot. Final reported accuracy/F1 are ordinary unweighted holdout
metrics, shown separately for each source.

In [ ]:
"""EfficientNetB0-only training/export helpers embedded into the Colab notebook."""
import hashlib
import json
import math
from pathlib import Path
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import (accuracy_score, average_precision_score, balanced_accuracy_score,
                             brier_score_loss, classification_report, confusion_matrix,
                             f1_score, precision_score, recall_score, roc_auc_score)


def balanced_training_frame(frame, source_weights, max_samples=16000, seed=42):
    """Balance classes AND source contributions; do not let PKLot drown out local data."""
    sources = sorted(frame.source.unique())
    total_weight = sum(source_weights.get(source, 1.0) for source in sources)
    target = min(max_samples, max(len(frame), 128))
    sampled = []
    for source in sources:
        weight = source_weights.get(source, 1.0)
        count = max(1, round(target * weight / total_weight / 2))
        for label in (0, 1):
            pool = frame[(frame.source == source) & (frame.label == label)]
            if pool.empty:
                raise ValueError(f'Training source {source} has no class {label}; collect more recording groups.')
            sampled.append(pool.sample(n=count, replace=count > len(pool), random_state=seed + label))
    return pd.concat(sampled, ignore_index=True).sample(frac=1, random_state=seed).reset_index(drop=True)


def validation_weights(frame, source_weights):
    weights = np.zeros(len(frame), dtype=np.float32)
    for source in frame.source.unique():
        for label in (0, 1):
            mask = (frame.source == source) & (frame.label == label)
            count = int(mask.sum())
            if count:
                weights[mask.to_numpy()] = source_weights.get(source, 1.0) / count / 2
    return weights / weights.mean()


def make_dataset(frame, batch_size=32, training=False, weights=None, seed=42):
    paths = frame.path.astype(str).to_numpy()
    labels = frame.label.to_numpy(dtype=np.float32)
    if weights is None:
        dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    else:
        dataset = tf.data.Dataset.from_tensor_slices((paths, labels, np.asarray(weights, dtype=np.float32)))
    if training:
        dataset = dataset.shuffle(min(len(frame), 4096), seed=seed, reshuffle_each_iteration=True)
    def decode(path, label, *sample_weight):
        image = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
        image.set_shape([None, None, 3])
        # Same RGB, bilinear resize, float32 and 0–255 contract as ml/worker.py.
        image = tf.image.resize(tf.cast(image, tf.float32), (224, 224), method='bilinear')
        return (image, label, sample_weight[0]) if sample_weight else (image, label)
    dataset = dataset.map(decode, num_parallel_calls=tf.data.AUTOTUNE)
    options = tf.data.Options()
    options.experimental_deterministic = not training
    return dataset.with_options(options).batch(batch_size).prefetch(tf.data.AUTOTUNE)


def build_vehicle_model(weights='imagenet', seed=42):
    inputs = tf.keras.Input((224, 224, 3), dtype=tf.float32, name='parking_crop_rgb')
    augmentation = tf.keras.Sequential([
        tf.keras.layers.RandomFlip('horizontal', seed=seed),
        tf.keras.layers.RandomRotation(0.025, fill_mode='reflect', seed=seed + 1),
        tf.keras.layers.RandomTranslation(0.04, 0.04, fill_mode='reflect', seed=seed + 2),
        tf.keras.layers.RandomZoom(0.10, fill_mode='reflect', seed=seed + 3),
        tf.keras.layers.RandomContrast(0.20, seed=seed + 4),
        tf.keras.layers.RandomBrightness(0.20, value_range=(0, 255), seed=seed + 5),
    ], name='training_augmentation')
    base = tf.keras.applications.EfficientNetB0(include_top=False, weights=weights, input_shape=(224, 224, 3), name='efficientnetb0')
    base.trainable = False
    x = base(augmentation(inputs), training=False)
    x = tf.keras.layers.GlobalAveragePooling2D(name='parking_pool')(x)
    x = tf.keras.layers.Dropout(0.30, seed=seed + 6, name='training_dropout')(x)
    output = tf.keras.layers.Dense(1, activation='sigmoid', name='vehicle_probability')(x)
    return tf.keras.Model(inputs, output, name='ParkVision_EfficientNetB0_Training')


def compile_vehicle_model(model, learning_rate):
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate, clipnorm=1.0),
                  loss=tf.keras.losses.BinaryCrossentropy(label_smoothing=0.02),
                  metrics=[tf.keras.metrics.BinaryAccuracy(name='accuracy', threshold=0.5),
                           tf.keras.metrics.Precision(name='precision'), tf.keras.metrics.Recall(name='recall'),
                           tf.keras.metrics.AUC(name='auc')], weighted_metrics=[])


def unfreeze_top_blocks(model):
    base = model.get_layer('efficientnetb0')
    base.trainable = True
    for layer in base.layers:
        layer.trainable = layer.name.startswith(('block6', 'block7', 'top_conv')) and not isinstance(layer, tf.keras.layers.BatchNormalization)
    return sum(layer.trainable for layer in base.layers)


def train_stages(train_dataset, validation_dataset, run_dir, head_epochs=5, fine_epochs=10, seed=42, weights='imagenet'):
    run_dir = Path(run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)
    def callbacks(stage):
        return [tf.keras.callbacks.ModelCheckpoint(str(run_dir / f'best_{stage}.keras'), monitor='val_loss', save_best_only=True),
                tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
                tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', patience=2, factor=0.5, min_lr=1e-7),
                tf.keras.callbacks.CSVLogger(str(run_dir / f'{stage}_history.csv')),
                tf.keras.callbacks.TerminateOnNaN()]
    model = build_vehicle_model(weights=weights, seed=seed)
    compile_vehicle_model(model, 1e-3)
    print('Stage 1: train the binary classification head; convolutional base frozen.')
    head = model.fit(train_dataset, validation_data=validation_dataset, epochs=head_epochs, callbacks=callbacks('head'))
    model = tf.keras.models.load_model(run_dir / 'best_head.keras', compile=False)
    print('Stage 2: fine-tune final EfficientNet blocks; BatchNorm stays frozen.')
    print('Trainable backbone layers:', unfreeze_top_blocks(model))
    compile_vehicle_model(model, 1e-5)
    fine = model.fit(train_dataset, validation_data=validation_dataset, epochs=fine_epochs, callbacks=callbacks('finetune'))
    candidates = [('head', min(head.history['val_loss'])), ('finetune', min(fine.history['val_loss']))]
    selected_stage, best_loss = min(candidates, key=lambda item: item[1])
    # Selection uses validation loss ONLY. Test results cannot change this choice.
    best = tf.keras.models.load_model(run_dir / f'best_{selected_stage}.keras', compile=False)
    info = {'selected_stage': selected_stage, 'validation_loss': float(best_loss),
            'head_epochs_run': len(head.history['loss']), 'fine_epochs_run': len(fine.history['loss']),
            'selection_metric': 'source-balanced validation loss'}
    return best, info, {'head': head.history, 'finetune': fine.history}


def predict_frame(model, frame, batch_size=32):
    dataset = make_dataset(frame, batch_size=batch_size)
    probabilities = np.asarray(model.predict(dataset, verbose=0)).reshape(-1)
    if len(probabilities) != len(frame) or not np.isfinite(probabilities).all():
        raise ValueError('Prediction count/values are invalid.')
    return probabilities


def binary_metrics(labels, probabilities):
    labels = np.asarray(labels, dtype=int)
    predictions = (np.asarray(probabilities) >= 0.5).astype(int)
    both_classes = len(np.unique(labels)) == 2
    return {'examples': int(len(labels)), 'unoccupied_examples': int((labels == 0).sum()),
            'occupied_examples': int((labels == 1).sum()), 'accuracy': float(accuracy_score(labels, predictions)),
            'balanced_accuracy': float(balanced_accuracy_score(labels, predictions)) if both_classes else None,
            'precision_occupied': float(precision_score(labels, predictions, zero_division=0)),
            'recall_occupied': float(recall_score(labels, predictions, zero_division=0)),
            'f1_occupied': float(f1_score(labels, predictions, zero_division=0)),
            'roc_auc': float(roc_auc_score(labels, probabilities)) if both_classes else None,
            'average_precision': float(average_precision_score(labels, probabilities)) if both_classes else None,
            'brier_score': float(brier_score_loss(labels, probabilities)),
            'confusion_matrix': confusion_matrix(labels, predictions, labels=[0, 1]).tolist(),
            'predicted_unoccupied': int((predictions == 0).sum()), 'predicted_occupied': int((predictions == 1).sum())}


def evaluate_holdout(model, manifest, run_dir, batch_size=32):
    run_dir = Path(run_dir)
    test = manifest[manifest.split == 'test'].reset_index(drop=True).copy()
    test['p_occupied'] = predict_frame(model, test, batch_size)
    test['prediction'] = (test.p_occupied >= 0.5).astype(int)
    report = {'threshold': 0.5, 'overall': binary_metrics(test.label, test.p_occupied), 'by_source': {}}
    for source, frame in test.groupby('source'):
        report['by_source'][source] = binary_metrics(frame.label, frame.p_occupied)
    print(classification_report(test.label, test.prediction, labels=[0, 1], target_names=['unoccupied', 'occupied'], zero_division=0))
    print('Per-source holdout results:')
    display_columns = ['examples', 'accuracy', 'precision_occupied', 'recall_occupied', 'f1_occupied']
    print(pd.DataFrame(report['by_source']).T[display_columns].to_string())
    if (test.prediction == test.prediction.iloc[0]).all():
        print('WARNING: The model predicted one class for the entire test set. Inspect data and do not claim it is validated.')
    test.to_csv(run_dir / 'test_predictions.csv', index=False)
    test[test.label != test.prediction].to_csv(run_dir / 'test_errors.csv', index=False)
    (run_dir / 'metrics.json').write_text(json.dumps(report, indent=2))
    fig, axis = plt.subplots(figsize=(5, 4))
    matrix = np.asarray(report['overall']['confusion_matrix'])
    axis.imshow(matrix, cmap='Blues')
    for (row, column), value in np.ndenumerate(matrix):
        axis.text(column, row, str(value), ha='center', va='center', color='white' if value > matrix.max() / 2 else 'black')
    axis.set(xticks=[0, 1], yticks=[0, 1], xticklabels=['unoccupied', 'occupied'], yticklabels=['unoccupied', 'occupied'],
             xlabel='Predicted', ylabel='Actual', title='Untouched holdout · EfficientNetB0')
    fig.tight_layout(); fig.savefig(run_dir / 'confusion_matrix.png', dpi=160); plt.show()
    return report, test


def inference_only_model(trained_model):
    """Reuse learned weights, discard augmentation/dropout; no custom layers needed."""
    inputs = tf.keras.Input((224, 224, 3), dtype=tf.float32, name='parking_crop_rgb')
    x = trained_model.get_layer('efficientnetb0')(inputs, training=False)
    x = trained_model.get_layer('parking_pool')(x)
    outputs = trained_model.get_layer('vehicle_probability')(x)
    return tf.keras.Model(inputs, outputs, name='ParkVision_EfficientNetB0_VehiclePresence')


def export_vehicle_model(trained_model, manifest, metrics, training_info, run_dir, config):
    run_dir = Path(run_dir)
    export_dir = run_dir / 'export'
    export_dir.mkdir(exist_ok=True)
    model = inference_only_model(trained_model)
    validation = manifest[manifest.split == 'val'].head(16)
    sample = next(iter(make_dataset(validation, batch_size=16)))[0]
    np.testing.assert_allclose(trained_model(sample, training=False).numpy(), model(sample, training=False).numpy(), rtol=1e-5, atol=1e-6)
    model_path = export_dir / 'parkvision_efficientnetb0.keras'
    model.save(model_path)
    loaded = tf.keras.models.load_model(model_path, compile=False)
    np.testing.assert_allclose(model(sample, training=False).numpy(), loaded(sample, training=False).numpy(), rtol=1e-5, atol=1e-6)
    local = metrics['by_source'].get('site')
    metadata = {'format_version': 1, 'model_name': model.name, 'model_version': run_dir.name,
                'class_names': ['unoccupied', 'occupied'], 'app_statuses': ['vacant', 'occupied'],
                'semantic_target': 'target vehicle present inside the supplied crop',
                'vehicle_categories': list(config['VEHICLE_CATEGORIES']), 'occupied_threshold': 0.5,
                'input_size': [224, 224], 'color_mode': 'RGB', 'input_dtype': 'float32', 'input_range': [0, 255],
                'resize_method': 'bilinear', 'preprocessing_inside_model': True,
                'tensorflow_version': tf.__version__, 'keras_version': tf.keras.__version__,
                'sha256': hashlib.sha256(model_path.read_bytes()).hexdigest(), 'training': training_info,
                'test_metrics': metrics, 'site_test_evaluated': bool(local and local['unoccupied_examples'] >= 5 and local['occupied_examples'] >= 5),
                'deployment_status': 'candidate_requires_site_validation',
                'save_reload_parity_passed': True, 'augmentation_removed': True}
    metadata_path = export_dir / 'parkvision_efficientnetb0.metadata.json'
    metadata_path.write_text(json.dumps(metadata, indent=2))
    (export_dir / 'README.txt').write_text(
        'ParkVision EfficientNetB0 binary vehicle-presence classifier\n'
        '0 = unoccupied (no target vehicle); 1 = occupied (target vehicle).\n'
        'No target vehicle does not mean physically unobstructed/parkable.\n'
        'This export is a candidate; inspect metrics and held-out site results.\n'
        'Copy BOTH model and metadata into ParkVision/ml/models, backing up the old pair.\n'
        'Restart ParkVision after replacing model files. The input stays RGB 224x224 float32, 0..255.\n'
        'Do not divide by 255; preprocessing is inside EfficientNet.\n'
        'The app review threshold is separate from the binary decision threshold (0.5).\n'
    )
    archive = run_dir / 'parkvision_efficientnetb0_export.zip'
    with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as bundle:
        for path in export_dir.iterdir():
            bundle.write(path, path.name)
        for name in ['metrics.json', 'test_predictions.csv', 'test_errors.csv', 'confusion_matrix.png', 'manifest.csv', 'training_info.json']:
            if (run_dir / name).exists():
                bundle.write(run_dir / name, name)
    return model, metadata, archive

In [ ]:
train_frame = manifest[manifest.split == 'train'].reset_index(drop=True)
val_frame = manifest[manifest.split == 'val'].reset_index(drop=True)
sampled_train = balanced_training_frame(train_frame, CONFIG['SOURCE_WEIGHTS'], CONFIG['MAX_TRAIN_SAMPLES_PER_EPOCH'], CONFIG['SEED'])
val_weights = validation_weights(val_frame, CONFIG['SOURCE_WEIGHTS'])
train_ds = make_dataset(sampled_train, CONFIG['BATCH_SIZE'], training=True, seed=CONFIG['SEED'])
val_ds = make_dataset(val_frame, CONFIG['BATCH_SIZE'], weights=val_weights)
sample_images, sample_labels = next(iter(train_ds))
assert sample_images.shape[1:] == (224, 224, 3)
assert 0 <= float(tf.reduce_min(sample_images)) <= float(tf.reduce_max(sample_images)) <= 255
print('Batch:', sample_images.shape, 'pixel range:', float(tf.reduce_min(sample_images)), float(tf.reduce_max(sample_images)))
print('Balanced training samples per epoch:', len(sampled_train))
display(sampled_train.groupby(['source', 'label']).size().rename('draws_per_epoch').reset_index())

## 8 · Train, then fine-tune (EfficientNetB0 only)

Stage 1 trains the head. Stage 2 unfreezes blocks 6/7 and the top convolution with
a smaller learning rate; BatchNorm stays frozen. Best checkpoints are written to
Drive after each improved epoch. The final candidate is selected **only by
validation loss**, never by its test score. Nothing is saved into your app yet.

In [ ]:
best_model, training_info, histories = train_stages(
    train_ds, val_ds, RUN_DIR,
    head_epochs=CONFIG['HEAD_EPOCHS'], fine_epochs=CONFIG['FINE_TUNE_EPOCHS'],
    seed=CONFIG['SEED'], weights='imagenet',
)
(RUN_DIR / 'training_info.json').write_text(json.dumps(training_info, indent=2))
(RUN_DIR / 'histories.json').write_text(json.dumps(histories, indent=2))
print('Chosen checkpoint:', training_info)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for stage, history in histories.items():
    axes[0].plot(history['loss'], label=f'{stage} train')
    axes[0].plot(history['val_loss'], '--', label=f'{stage} validation')
    axes[1].plot(history['accuracy'], label=f'{stage} train')
    axes[1].plot(history['val_accuracy'], '--', label=f'{stage} validation')
axes[0].set_title('Loss (source-balanced validation)'); axes[1].set_title('Unweighted accuracy')
for axis in axes:
    axis.set_xlabel('Epoch within stage'); axis.legend()
fig.tight_layout(); fig.savefig(RUN_DIR / 'training_curves.png', dpi=160); plt.show()

## 9 · Evaluate the untouched test groups

Decision threshold stays **0.5**, matching ParkVision. Do not adjust it after
looking at test errors. A high public-data score is not evidence of night-camera
accuracy. The table below explicitly shows whether site test data exists.

In [ ]:
metrics, test_predictions = evaluate_holdout(best_model, manifest, RUN_DIR, CONFIG['BATCH_SIZE'])
display(pd.DataFrame(metrics['by_source']).T)
if 'site' not in metrics['by_source']:
    print('SITE VALIDATION: NOT MEASURED. Add independent mounted-phone recordings before claiming Luzano accuracy.')
else:
    print('SITE TEST:', metrics['by_source']['site'])
errors = test_predictions[test_predictions.label != test_predictions.prediction].copy()
if len(errors):
    errors['wrong_score'] = np.where(errors.prediction == 1, errors.p_occupied, 1 - errors.p_occupied)
    examples = errors.sort_values('wrong_score', ascending=False).head(12)
    fig, axes = plt.subplots(3, 4, figsize=(11, 8))
    for axis in axes.ravel():
        axis.axis('off')
    for axis, row in zip(axes.ravel(), examples.to_dict('records')):
        with Image.open(row['path']) as image:
            axis.imshow(image)
        axis.set_title(f"{row['source']} true={CLASS_NAMES[row['label']]}\np(occupied)={row['p_occupied']:.3f}", fontsize=8)
    fig.suptitle('Hard errors: inspect them; keep them in the evaluation')
    fig.tight_layout(); fig.savefig(RUN_DIR / 'hard_errors.png', dpi=160); plt.show()

## 10 · Export and download the app-ready package

The export reuses the learned weights but removes training augmentation/dropout.
It checks inference parity before/after export and after reloading `.keras`.
**Do not use `tf.saved_model.save` here.** One ZIP contains the model, metadata,
metrics, confusion matrix and split manifest. The ZIP is also preserved on Drive.

In [ ]:
inference_model, metadata, export_zip = export_vehicle_model(
    best_model, manifest, metrics, training_info, RUN_DIR, CONFIG,
)
print('EXPORT:', export_zip)
print('MODEL:', metadata['model_name'], 'SHA256:', metadata['sha256'])
print('Save/reload parity:', metadata['save_reload_parity_passed'])
print('Site test evaluated:', metadata['site_test_evaluated'])
print('This is a candidate. Review the actual test results before replacing the app model.')
from google.colab import files
files.download(str(export_zip))
EXPORT_DONE_THIS_SESSION = True

## 11 · Put it into ParkVision

Unzip the download. The two files that belong together are:

- `parkvision_efficientnetb0.keras`
- `parkvision_efficientnetb0.metadata.json`

Back up the previous model, then place both files into `ParkVision/ml/models/`.
Restart `npm run dev` so Python loads the new weights. **No SQL reset or new
Supabase tables are needed.** The worker still expects RGB 224×224 float32, 0–255,
with output 0 = app vacant, 1 = occupied. Do not add another `/255` operation.

The app's `MINIMUM_CONFIDENCE=0.80` is a separate publication/review filter, not the
binary decision threshold or a measured accuracy guarantee. Keep model evaluation
separate from whether the UI permits publication.

## 12 · Optional: test your actual photo (does not train on it)

Set `RUN_MANUAL_IMAGE_TEST=True` and rerun this cell. Upload one photo and edit
`TEST_BOXES` to your normalized x/y/width/height values. Leave out the building
ceiling where practical, but keep a fixed parking-region box when a car leaves.
The default rectangles are illustrative, not a guaranteed calibration.

In [ ]:
if CONFIG['RUN_MANUAL_IMAGE_TEST']:
    from google.colab import files
    uploaded = files.upload()
    name = next(iter(uploaded))
    image = ImageOps.exif_transpose(Image.open(name)).convert('RGB')
    TEST_BOXES = [
        {'name': 'left position', 'x': 0.08, 'y': 0.45, 'width': 0.38, 'height': 0.50},
        {'name': 'right position', 'x': 0.46, 'y': 0.40, 'width': 0.53, 'height': 0.48},
    ]
    crops = []
    for box in TEST_BOXES:
        pixel_box = clean_bbox([box['x'] * image.width, box['y'] * image.height,
                                box['width'] * image.width, box['height'] * image.height], image.width, image.height)
        if pixel_box is None:
            raise ValueError('Invalid test rectangle.')
        crops.append(tf.image.resize(np.asarray(image.crop(pixel_box), dtype=np.float32), (224, 224), method='bilinear'))
    scores = np.asarray(inference_model(tf.stack(crops), training=False)).reshape(-1)
    fig, axes = plt.subplots(1, len(crops), figsize=(5 * len(crops), 4), squeeze=False)
    for axis, box, crop, score in zip(axes.ravel(), TEST_BOXES, crops, scores):
        axis.imshow(np.clip(crop.numpy(), 0, 255).astype('uint8'))
        axis.set_title(f"{box['name']}\n{CLASS_NAMES[int(score >= 0.5)]}, p(occupied)={score:.3f}")
        axis.axis('off')
    fig.tight_layout(); plt.show()
else:
    print('Skipped optional photo upload. The model ZIP has already been exported.')

## Recovery after Colab disconnects

If an export completed, run **only the cell below** in a new session to download
the latest saved ZIP. It does not retrain. If training stopped before export, the
Drive run folder still contains `best_head.keras` / `best_finetune.keras` and logs.
Restore that checkpoint instead of rebuilding an untrained head. A partially
trained checkpoint still needs its held-out evaluation before deployment.

In [ ]:
# Independent recovery cell. No training variables are needed.
from pathlib import Path
if globals().get('EXPORT_DONE_THIS_SESSION', False):
    print('Normal run completed; the export was already downloaded. Recovery is only needed in a new session.')
else:
    from google.colab import drive, files
    drive.mount('/content/drive')
    saved_exports = sorted(Path('/content/drive/MyDrive/ParkVision_EfficientNetB0/runs').glob('*/parkvision_efficientnetb0_export.zip'))
    if saved_exports:
        print('Downloading saved export:', saved_exports[-1])
        files.download(str(saved_exports[-1]))
    else:
        checkpoints = sorted(Path('/content/drive/MyDrive/ParkVision_EfficientNetB0/runs').glob('*/best_*.keras'))
        print('No completed export found. Saved checkpoints:')
        for checkpoint in checkpoints:
            print(checkpoint)
        print('Weights in these .keras checkpoints are preserved; do not overwrite them by rebuilding an untrained model.')

## Sources and reporting

- [Keras EfficientNet fine-tuning](https://keras.io/examples/vision/image_classification_efficientnet_fine_tuning/)
- [Keras transfer learning / BatchNorm guidance](https://keras.io/guides/transfer_learning/)
- [Keras model saving](https://keras.io/api/models/model_saving_apis/model_saving_and_loading/)
- [PKLot dataset / citation and license](https://web.inf.ufpr.br/vri/databases/parking-lot-database/)
- [COCO dataset / terms](https://cocodataset.org/#termsofuse)

Public datasets retain their original licenses. Cite the datasets in your report.
Report this as a **new EfficientNetB0 experiment**, not the old S2 comparison score.
Do not copy the old 96.88% result onto newly trained weights. Use the metrics saved
by this run, and explicitly state when local day/night accuracy was not measured.